# New utensil-only specialist — 20 epochs, moderate augmentations

**Experimental YOLO11m specialist.** Does **not** modify your frozen V1 utensil checkpoint or V4 model.

Uses existing V4 photographs and extracts **only utensil annotations**, making a one-class dataset. Primary training: 521 cafeteria images containing 294 labeled utensils (154 photos have positive labels). Optional 161 legacy photos add 244 utensil annotations (128 have utensil labels), but these photos differ from cafeteria scenes; default is **off**.

Validation: unchanged 97 photos with 48 utensil boxes. **Test never used.**

Moderate augmentations: rotation ±10°, translation 8%, scale ±25%, saturation .35, brightness .25, mosaic .5. Input 800px, 20 epochs maximum, patience 10. Labels are provisional; unannotated utensils may be mistaken for background. **Audit negatives before interpreting this as final performance.** Select a T4 GPU.


## 1. Install and load training script


In [ ]:
import subprocess,sys,torch,hashlib,zipfile,json
from pathlib import Path
from io import BytesIO
from google.colab import files
assert torch.cuda.is_available(),"Switch Runtime to T4 GPU first"
subprocess.run([sys.executable,"-m","pip","install","-q","ultralytics==8.4.175"],check=True)
repo=Path("/content/yolo-cafeteria-research")
if repo.exists():
    subprocess.run(["git","-C",str(repo),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1","https://github.com/cx-57/yolo-cafeteria-research.git",str(repo)],check=True)
trainer=repo/"experiments/yolo-v4-4class/utensil_specialist_20epoch.py"
assert trainer.is_file()
print("GPU:",torch.cuda.get_device_name(0),flush=True)


## 2. Upload existing 151 MB V4 dataset ZIP


In [ ]:
name="YOLO_V4_865_4class_audit_pending.zip"
root=Path("/content/yolo_v4_four_class")
if not (root/"source_manifest.json").is_file():
    print("Upload",name,flush=True)
    upload=files.upload()
    assert name in upload,"Wrong ZIP"
    raw=upload[name]
    assert hashlib.sha256(raw).hexdigest()=="95a386dbd9cf707e642237b92326a27ffd4958b893d1e0a7089e463b823c91e7","ZIP mismatch"
    with zipfile.ZipFile(BytesIO(raw)) as z:
        assert z.testzip() is None
        assert all(Path(i.filename).parts[:1]==("yolo_v4_four_class",) for i in z.infolist())
        z.extractall("/content")
assert (root/"source_manifest.json").is_file()
print("Dataset loaded. No test images are used.",flush=True)


## 3. Train one new utensil specialist, separate from existing models


In [ ]:
# Default False: test whether cafeteria-only data helps before adding isolated cutlery.
INCLUDE_LEGACY=False
out=Path("/content/V4_new_utensil_specialist_"+("legacy" if INCLUDE_LEGACY else "cafeteria"))
assert not out.exists(),"Result folder exists; use a new runtime or a new output folder"
cmd=[sys.executable,"-u",str(trainer),"--root",str(root),"--output",str(out),
     "--device","0","--batch","4"]
if INCLUDE_LEGACY: cmd.append("--include-legacy")
subprocess.run(cmd,check=True)


## 4. Review validation AP50 and AP50–95, download checkpoint


In [ ]:
report=out/"utensil_val_metrics.json"
assert report.is_file(),"Training and validation not completed"
print(json.dumps(json.loads(report.read_text()),indent=2))
result_zip=Path("/content/V4_new_utensil_specialist_results.zip")
with zipfile.ZipFile(result_zip,"w",zipfile.ZIP_DEFLATED) as z:
    for rel in ["train/weights/best.pt","train/results.csv",
                "utensil_val_metrics.json","source_counts.json"]:
        f=out/rel
        if f.is_file():z.write(f,arcname=rel)
files.download(str(result_zip))
